# Qwen 7B fine-tuned evaluation

Run the five-item preview first, review it, then run all 60 questions with the same reranked retriever.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil
drive.mount('/content/drive')
REPO=Path('/content/farmer-llm-project')
os.chdir(REPO)
os.environ['FARMER_MODEL_ID']='Qwen/Qwen2.5-7B-Instruct'
ADAPTER='/content/drive/MyDrive/farmer-llm-7b/run1/adapter'
print('Model:', os.environ['FARMER_MODEL_ID'])
print('Adapter:', ADAPTER)

In [ ]:
%pip install -r requirements-colab.txt

In [ ]:
%%time
!python src/run_conditions.py --conditions finetuned,finetuned_rag --limit 5 --adapter /content/drive/MyDrive/farmer-llm-7b/run1/adapter --out results/qwen7b_reranked --rerank --candidate-k 20
for condition in ['finetuned','finetuned_rag']:
    print('\n###', condition)
    with open(f'results/qwen7b_reranked/raw_{condition}.jsonl', encoding='utf-8') as f:
        for line in f: print(json.loads(line)['id'], json.loads(line)['answer'])

## Review q001–q005 before continuing

In [ ]:
%%time
!python src/run_conditions.py --conditions finetuned,finetuned_rag --adapter /content/drive/MyDrive/farmer-llm-7b/run1/adapter --out results/qwen7b_reranked --rerank --candidate-k 20 --resume
for path in Path('results/qwen7b_reranked').glob('raw_*.jsonl'): shutil.copy2(path, Path('/content/drive/MyDrive/farmer-llm-7b') / path.name)
print('Copied Qwen 7B outputs to Drive')